In [3]:
import joblib
import numpy as np
import os

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

In [4]:
save_dir = "data/processed"

raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
temperature = np.load(os.path.join(save_dir, "temperature.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
scaler      = joblib.load(os.path.join(save_dir, "scaler.pkl"))

num_train_samples = len(train_idx)
num_val_samples   = len(val_idx)
num_test_samples  = len(test_idx)

temp_mean = scaler.mean_[1]
temp_std  = scaler.scale_[1]

print(f"raw_data:    {raw_data.shape}")
print(f"temperature: {temperature.shape}")
print(f"train/val/test: {num_train_samples} / {num_val_samples} / {num_test_samples}")


raw_data:    (420451, 14)
temperature: (420451,)
train/val/test: 210225 / 105112 / 105114


In [12]:
sampling_rate   = 6
sequence_length = 120
delay           = sampling_rate * (sequence_length + 24 - 1)
batch_size      = 256

class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Valid starting indices: each sequence of length sequence_length
        # sampled every sampling_rate steps needs:
        # (sequence_length - 1) * sampling_rate + 1 rows ahead
        self.indices = np.arange(start_index, end_index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Sample every `sampling_rate` steps for `sequence_length` steps
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # Target is `delay` steps ahead of the sequence start
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [13]:
train_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train_samples,
)

val_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train_samples, end_index=num_train_samples + num_val_samples,
)

test_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train_samples + num_val_samples, end_index=len(raw_data) - delay,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# Quick sanity check
for inputs, targets in train_loader:
    print("Input shape:", inputs.shape)   # (batch_size, sequence_length, num_features)
    print("Target shape:", targets.shape) # (batch_size,)
    break

Input shape: torch.Size([256, 120, 14])
Target shape: torch.Size([256])


In [14]:
def evaluate_naive_method(loader):
    total_abs_err = 0.0
    samples_seen  = 0
    with torch.no_grad():
        for samples, targets in loader:
            preds = samples[:, -1, 1] * temp_std + temp_mean  # un-normalize col 1
            # targets already in °C — no conversion
            total_abs_err += torch.sum(torch.abs(preds - targets)).item()
            samples_seen  += samples.shape[0]
    return total_abs_err / samples_seen

In [15]:
print(f"Validation MAE: {evaluate_naive_method(val_loader):.2f}")
print(f"Test MAE:       {evaluate_naive_method(test_loader):.2f}")

Validation MAE: 2.44
Test MAE:       2.62
